# Step 2 — Analyze Patterns

1. Run the **list cell** to see all available saved models.
2. Set `MODEL_TO_LOAD` in the **configuration cell**.
3. Adjust the pattern toggles, then run all cells.

In [ ]:
# ============================================================
# CONFIGURATION — set MODEL_TO_LOAD, then run all cells
# ============================================================

# Folder name inside output/ — run the listing cell below to see options
MODEL_TO_LOAD = "default__english__20260604_143022"

# Which label columns to cross-tabulate with topics
LABEL_COLUMNS = ["gender", "body_part"]

# NLP scorer modules to apply (each must live in nlp_scripts/ and expose score(text)->float)
NLP_SCORE_SCRIPTS = ["hedge_word_counter", "pi_risk_estimator"]

# Toggle individual analyses
RUN_LABEL_ANALYSIS = True
RUN_SCORE_ANALYSIS = True

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import importlib
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_VIZ = True
except ImportError:
    HAS_VIZ = False
    print("Install matplotlib and seaborn for charts.")

In [ ]:
# List all saved models — run this cell to pick a value for MODEL_TO_LOAD above
output_dir = "output"
print(f"{'Run name':<45} {'embedding':<20} {'stopwords':<22} {'topics':>6} {'docs':>8}  trained")
print("-" * 120)
if os.path.isdir(output_dir):
    for name in sorted(os.listdir(output_dir)):
        cfg_path = os.path.join(output_dir, name, "run_config.json")
        if os.path.exists(cfg_path):
            with open(cfg_path) as f:
                c = json.load(f)
            print(f"{name:<45} {c.get('embedding','?'):<20} {c.get('stopwords','?'):<22}"
                  f" {c.get('n_topics','?'):>6} {c.get('n_docs','?'):>8}  {c.get('trained_on','?')[:10]}")
else:
    print("No output/ directory found — run 01_build_model.ipynb first.")

In [ ]:
run_dir = os.path.join("output", MODEL_TO_LOAD)
cfg_path = os.path.join(run_dir, "run_config.json")

with open(cfg_path) as f:
    run_cfg = json.load(f)

print(f"Loading model: {MODEL_TO_LOAD}")
print(json.dumps(run_cfg, indent=2))

# Re-create the embedding model that was used during training
if run_cfg["embedding"] == "clinical_radiology":
    embedding_model = SentenceTransformer("emilyalsentzer/Bio_ClinicalBERT")
else:
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic.load(os.path.join(run_dir, "model"), embedding_model=embedding_model)
df = pd.read_parquet(os.path.join(run_dir, "annotated_data.parquet"))
print(f"\nLoaded {len(df)} records. Topics: {sorted(df['topic'].unique())}")
display(df.head())

In [ ]:
# === Label Analysis ===
if RUN_LABEL_ANALYSIS:
    print("=== Topic-Label Relationships ===\n")
    for col in LABEL_COLUMNS:
        if col not in df.columns:
            print(f"  Column '{col}' not found in data — skipping.")
            continue
        dist = df.groupby(["topic", col]).size().unstack(fill_value=0)
        print(f"Distribution of '{col}' across topics:")
        display(dist)
        if HAS_VIZ:
            ax = dist.plot(kind="bar", stacked=True, figsize=(8, 4),
                           title=f"Topic distribution by {col}")
            ax.set_xlabel("Topic")
            ax.set_ylabel("Count")
            plt.tight_layout()
            plt.show()
else:
    print("Label analysis is disabled (RUN_LABEL_ANALYSIS = False).")

In [ ]:
# === NLP Score Analysis ===
if RUN_SCORE_ANALYSIS:
    print("=== NLP Score Distributions per Topic ===\n")
    active_scores = []
    for script_name in NLP_SCORE_SCRIPTS:
        try:
            mod = importlib.import_module(f"nlp_scripts.{script_name}")
            df[script_name] = df["text"].apply(mod.score)
            active_scores.append(script_name)
            print(f"  Applied: {script_name}")
        except (ImportError, AttributeError) as e:
            print(f"  Warning: could not load '{script_name}': {e}")

    if active_scores:
        print("\nMean scores per topic:")
        display(df.groupby("topic")[active_scores].mean().round(3))

        if HAS_VIZ:
            for score_col in active_scores:
                fig, ax = plt.subplots(figsize=(8, 4))
                df.boxplot(column=score_col, by="topic", ax=ax)
                ax.set_title(f"{score_col} distribution per topic")
                ax.set_xlabel("Topic")
                ax.set_ylabel(score_col)
                plt.suptitle("")
                plt.tight_layout()
                plt.show()
else:
    print("Score analysis is disabled (RUN_SCORE_ANALYSIS = False).")